# Notebook 3A — Modelagem preditiva das emissões horárias de CO₂

## Carga instantânea × contexto e trajetória operacional

Pergunta experimental: informações sobre o contexto e a trajetória operacional melhoram a estimação das emissões horárias de CO₂ em relação ao uso apenas da carga instantânea?

Cenário A: somente carga atual P(t).

Cenário B: carga atual + memória e trajetória operacional.

A comparação preserva a ordem temporal: janeiro a setembro para treino, outubro para validação e novembro a dezembro para teste.


In [ ]:
# ============================================================
# CÉLULA 1 — CONFIGURAÇÃO E CARREGAMENTO
# ============================================================

from google.colab import drive
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

drive.mount("/content/drive")

PROJECT_SLUG = "co2-cems-thermal-modeling"
ROOT = Path("/content/drive/MyDrive") / PROJECT_SLUG

DATA_PROCESSED = ROOT / "data" / "processed"
PIPELINE_DIR = ROOT / "pipeline"
OUTPUT = ROOT / "outputs"
REPORTS = OUTPUT / "reports"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"
MODELS = OUTPUT / "models"

for folder in [REPORTS, FIGURES, TABLES, MODELS]:
    folder.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = PIPELINE_DIR / "selected_unit.json"
MODEL_DATA_PATH = DATA_PROCESSED / "modeling_dataset_operating_hours.parquet"

if not CONFIG_PATH.exists():
    raise FileNotFoundError("selected_unit.json não encontrado. Execute o Notebook 1.")

if not MODEL_DATA_PATH.exists():
    raise FileNotFoundError(
        "modeling_dataset_operating_hours.parquet não encontrado. "
        "Execute o Notebook 2."
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    selected_unit = json.load(f)

model_data = pd.read_parquet(MODEL_DATA_PATH)
model_data["timestamp"] = pd.to_datetime(model_data["timestamp"], errors="coerce")
model_data = model_data.sort_values("timestamp").reset_index(drop=True)

print("=" * 72)
print("FASE 3A — MODELAGEM PREDITIVA DE CO2")
print("=" * 72)
print(f"Unidade: {selected_unit['facility_name']} — Unit {selected_unit['unit_id']}")
print(f"Ano: {selected_unit['year']}")
print(f"Observações operacionais: {len(model_data):,}")
print(f"Período: {model_data['timestamp'].min()} → {model_data['timestamp'].max()}")
print("Interface Notebook 2 → Notebook 3A: OK")


In [ ]:
# ============================================================
# CÉLULA 2 — CENÁRIOS E DIVISÃO TEMPORAL
# ============================================================

TARGET = "CO2 Mass (short tons)"

FEATURES_A = [
    "load_current_mw"
]

FEATURES_B = [
    "load_current_mw",
    "load_lag_1h",
    "delta_load_1h",
    "abs_delta_load_1h",
    "load_mean_previous_3h",
    "operating_time_lag_1h",
    "hours_since_startup"
]

required = ["timestamp", TARGET] + sorted(set(FEATURES_A + FEATURES_B))
missing = [col for col in required if col not in model_data.columns]

if missing:
    raise ValueError(f"Colunas ausentes: {missing}")

experiment_data = (
    model_data[required]
    .dropna()
    .sort_values("timestamp")
    .reset_index(drop=True)
)

YEAR = int(selected_unit["year"])
TRAIN_END = pd.Timestamp(f"{YEAR}-09-30 23:00:00")
VALIDATION_END = pd.Timestamp(f"{YEAR}-10-31 23:00:00")

train_data = experiment_data[
    experiment_data["timestamp"] <= TRAIN_END
].copy()

validation_data = experiment_data[
    (experiment_data["timestamp"] > TRAIN_END)
    & (experiment_data["timestamp"] <= VALIDATION_END)
].copy()

test_data = experiment_data[
    experiment_data["timestamp"] > VALIDATION_END
].copy()

split_summary = pd.DataFrame({
    "conjunto": ["Treino", "Validação", "Teste"],
    "inicio": [
        train_data["timestamp"].min(),
        validation_data["timestamp"].min(),
        test_data["timestamp"].min()
    ],
    "fim": [
        train_data["timestamp"].max(),
        validation_data["timestamp"].max(),
        test_data["timestamp"].max()
    ],
    "observacoes": [
        len(train_data),
        len(validation_data),
        len(test_data)
    ]
})

split_summary["percentual"] = (
    100 * split_summary["observacoes"] / len(experiment_data)
)

print("Cenário A:", FEATURES_A)
print("Cenário B:", FEATURES_B)
print("Alvo:", TARGET)
display(split_summary.round(2))


In [ ]:
# ============================================================
# CÉLULA 3 — NORMALIZAÇÃO SEM DATA LEAKAGE
# ============================================================
# Todos os scalers são ajustados somente no treino.

X_train_A = train_data[FEATURES_A].copy()
X_val_A = validation_data[FEATURES_A].copy()
X_test_A = test_data[FEATURES_A].copy()

X_train_B = train_data[FEATURES_B].copy()
X_val_B = validation_data[FEATURES_B].copy()
X_test_B = test_data[FEATURES_B].copy()

y_train = train_data[[TARGET]].copy()
y_val = validation_data[[TARGET]].copy()
y_test = test_data[[TARGET]].copy()

scaler_X_A = MinMaxScaler()
scaler_X_B = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_A_s = scaler_X_A.fit_transform(X_train_A)
X_val_A_s = scaler_X_A.transform(X_val_A)
X_test_A_s = scaler_X_A.transform(X_test_A)

X_train_B_s = scaler_X_B.fit_transform(X_train_B)
X_val_B_s = scaler_X_B.transform(X_val_B)
X_test_B_s = scaler_X_B.transform(X_test_B)

y_train_s = scaler_y.fit_transform(y_train).ravel()
y_val_s = scaler_y.transform(y_val).ravel()

print("Normalização concluída.")
print("Treino A:", X_train_A_s.shape)
print("Treino B:", X_train_B_s.shape)


In [ ]:
# ============================================================
# CÉLULA 4 — MLP E TREINAMENTO CONTROLADO
# ============================================================
# Mesma arquitetura e mesmos hiperparâmetros nos dois cenários.

HIDDEN_LAYERS = (64, 32)
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
MAX_EPOCHS = 500
PATIENCE = 30

def build_mlp(n_features, seed=SEED):
    tf.keras.utils.set_random_seed(seed)

    model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="linear")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="mse"
    )

    return model

def train_model(X_train, y_train, X_val, y_val, n_features):
    model = build_mlp(n_features)

    callback = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        min_delta=1e-6,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=False,
        callbacks=[callback],
        verbose=0
    )

    return model, history

print("Treinando Cenário A...")
model_A, history_A = train_model(
    X_train_A_s, y_train_s,
    X_val_A_s, y_val_s,
    len(FEATURES_A)
)

print("Treinando Cenário B...")
model_B, history_B = train_model(
    X_train_B_s, y_train_s,
    X_val_B_s, y_val_s,
    len(FEATURES_B)
)

best_epoch_A = int(np.argmin(history_A.history["val_loss"]) + 1)
best_epoch_B = int(np.argmin(history_B.history["val_loss"]) + 1)

print("Melhor época A:", best_epoch_A)
print("Melhor época B:", best_epoch_B)
print("Parâmetros A:", model_A.count_params())
print("Parâmetros B:", model_B.count_params())


In [ ]:
# ============================================================
# CÉLULA 5 — MÉTRICAS NO TESTE FINAL
# ============================================================

def inverse_y(values):
    return scaler_y.inverse_transform(
        np.asarray(values).reshape(-1, 1)
    ).ravel()

def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

pred_A = inverse_y(
    model_A.predict(X_test_A_s, verbose=0).ravel()
)

pred_B = inverse_y(
    model_B.predict(X_test_B_s, verbose=0).ravel()
)

y_true = y_test[TARGET].to_numpy()

metrics_A = metrics(y_true, pred_A)
metrics_B = metrics(y_true, pred_B)

results = pd.DataFrame([
    {
        "cenario": "A — carga instantânea",
        "n_features": len(FEATURES_A),
        **metrics_A
    },
    {
        "cenario": "B — carga + trajetória",
        "n_features": len(FEATURES_B),
        **metrics_B
    }
])

mae_gain = 100 * (
    metrics_A["MAE"] - metrics_B["MAE"]
) / metrics_A["MAE"]

rmse_gain = 100 * (
    metrics_A["RMSE"] - metrics_B["RMSE"]
) / metrics_A["RMSE"]

delta_r2 = metrics_B["R2"] - metrics_A["R2"]

display(results.round(4))

print(f"Redução de MAE A → B: {mae_gain:+.2f}%")
print(f"Redução de RMSE A → B: {rmse_gain:+.2f}%")
print(f"Variação de R² A → B: {delta_r2:+.4f}")

if (
    metrics_B["MAE"] < metrics_A["MAE"]
    and metrics_B["RMSE"] < metrics_A["RMSE"]
):
    print("Conclusão preliminar: B reduziu os erros globais no teste.")
else:
    print("Conclusão preliminar: B não reduziu simultaneamente MAE e RMSE.")


In [ ]:
# ============================================================
# CÉLULA 6 — ANÁLISE POR CONTEXTO OPERACIONAL
# ============================================================
# Avalia startup e rampas intensas, situações em que P(t)
# tende a representar menos completamente o estado da unidade.

analysis_test = test_data[[
    "timestamp",
    "load_current_mw",
    "abs_delta_load_1h",
    "hours_since_startup",
    TARGET
]].copy()

analysis_test["pred_A"] = pred_A
analysis_test["pred_B"] = pred_B

analysis_test["abs_error_A"] = np.abs(
    analysis_test[TARGET] - analysis_test["pred_A"]
)

analysis_test["abs_error_B"] = np.abs(
    analysis_test[TARGET] - analysis_test["pred_B"]
)

ramp_threshold = train_data[
    "abs_delta_load_1h"
].quantile(0.90)

contexts = {
    "Todas as horas de teste":
        pd.Series(True, index=analysis_test.index),

    "Primeiras 6 h após startup":
        (
            (analysis_test["hours_since_startup"] >= 1)
            & (analysis_test["hours_since_startup"] <= 6)
        ),

    "Rampas intensas (≥ P90 treino)":
        analysis_test["abs_delta_load_1h"] >= ramp_threshold
}

rows = []

for name, mask in contexts.items():
    subset = analysis_test.loc[mask]

    if subset.empty:
        continue

    mae_A_ctx = subset["abs_error_A"].mean()
    mae_B_ctx = subset["abs_error_B"].mean()

    rows.append({
        "contexto": name,
        "n": len(subset),
        "MAE_A": mae_A_ctx,
        "MAE_B": mae_B_ctx,
        "redução_MAE_%":
            100 * (mae_A_ctx - mae_B_ctx) / mae_A_ctx
    })

context_results = pd.DataFrame(rows)

print(
    f"Limiar de rampa intensa: {ramp_threshold:.2f} MW/h"
)
display(context_results.round(3))


In [ ]:
# ============================================================
# CÉLULA 7 — VISUALIZAÇÕES
# ============================================================

COLOR_BG = "#F6F1EA"
COLOR_TEXT = "#2F2A26"
COLOR_ORANGE = "#D9913B"
COLOR_RED = "#B6543A"
COLOR_GRID = "#D8D0C7"

plt.rcParams.update({
    "figure.facecolor": COLOR_BG,
    "axes.facecolor": COLOR_BG,
    "text.color": COLOR_TEXT,
    "axes.labelcolor": COLOR_TEXT,
    "grid.color": COLOR_GRID
})

# Curvas de validação
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    history_A.history["val_loss"],
    label="Cenário A",
    color=COLOR_ORANGE
)

ax.plot(
    history_B.history["val_loss"],
    label="Cenário B",
    color=COLOR_RED
)

ax.set_title("Curvas de validação", loc="left")
ax.set_xlabel("Época")
ax.set_ylabel("MSE normalizado")
ax.grid(alpha=0.4)
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(
    FIGURES / "03A_curvas_validacao.png",
    dpi=220,
    bbox_inches="tight"
)
plt.show()

# Observado x previsto
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    test_data["timestamp"],
    y_true,
    label="Observado",
    color=COLOR_TEXT,
    linewidth=1.6
)

ax.plot(
    test_data["timestamp"],
    pred_A,
    label="Cenário A",
    color=COLOR_ORANGE,
    linewidth=1.1
)

ax.plot(
    test_data["timestamp"],
    pred_B,
    label="Cenário B",
    color=COLOR_RED,
    linewidth=1.1
)

ax.set_title(
    "Emissões observadas e estimadas no teste",
    loc="left"
)
ax.set_ylabel("CO₂ Mass [short tons]")
ax.set_xlabel("Tempo")
ax.grid(alpha=0.3)
ax.legend(frameon=False, ncol=3)

fig.tight_layout()
fig.savefig(
    FIGURES / "03A_observado_vs_previsto.png",
    dpi=220,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# ============================================================
# CÉLULA 8 — EXPORTAÇÃO E RELATÓRIO
# ============================================================

import joblib
from IPython.display import display, Markdown

results.to_csv(
    TABLES / "03A_model_comparison.csv",
    index=False
)

context_results.to_csv(
    TABLES / "03A_context_comparison.csv",
    index=False
)

analysis_test[[
    "timestamp",
    TARGET,
    "pred_A",
    "pred_B",
    "abs_error_A",
    "abs_error_B"
]].to_csv(
    TABLES / "03A_test_predictions.csv",
    index=False
)

model_A.save(
    MODELS / "03A_mlp_cenario_A.keras"
)

model_B.save(
    MODELS / "03A_mlp_cenario_B.keras"
)

joblib.dump(
    scaler_X_A,
    MODELS / "03A_scaler_X_A.joblib"
)

joblib.dump(
    scaler_X_B,
    MODELS / "03A_scaler_X_B.joblib"
)

joblib.dump(
    scaler_y,
    MODELS / "03A_scaler_y.joblib"
)

report = f"""
# Relatório — Notebook 3A

## Pergunta experimental

A inclusão de informações sobre contexto e trajetória operacional melhora a estimação das emissões horárias de CO₂ em relação ao uso apenas da carga instantânea?

## Cenários

**A — carga instantânea:** 1 feature.

**B — carga + trajetória operacional:** 7 features.

## Divisão temporal

- Treino: janeiro a setembro — {len(train_data):,} observações
- Validação: outubro — {len(validation_data):,} observações
- Teste: novembro e dezembro — {len(test_data):,} observações

## Arquitetura

MLP: entrada → Dense(64, ReLU) → Dense(32, ReLU) → Dense(1, linear)

## Resultados no teste

| Cenário | MAE | RMSE | R² |
|---|---:|---:|---:|
| A | {metrics_A['MAE']:.4f} | {metrics_A['RMSE']:.4f} | {metrics_A['R2']:.4f} |
| B | {metrics_B['MAE']:.4f} | {metrics_B['RMSE']:.4f} | {metrics_B['R2']:.4f} |

Redução de MAE A → B: {mae_gain:+.2f}%

Redução de RMSE A → B: {rmse_gain:+.2f}%

Variação de R² A → B: {delta_r2:+.4f}

## Interpretação

A hipótese deve ser aceita apenas se o Cenário B apresentar ganho consistente no conjunto de teste. A análise por contexto operacional complementa as métricas globais ao verificar o comportamento do erro durante startups e rampas intensas.
"""

display(Markdown(report))

REPORT_PATH = REPORTS / "relatorio_modelagem_preditiva_3A.md"

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(report)

print("Relatório salvo em:", REPORT_PATH)
print("NOTEBOOK 3A CONCLUÍDO")
